# Naiive clustering

This notebook performs the naiive clustering with genome length and gc_content only

This notebbok is simplified version of '3. run_clustering.ipynb'

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from Bio import Entrez, SeqIO
import time
from tqdm.auto import tqdm
import re
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df_full = pd.read_csv('embeddings.csv')

df_full.describe().T

In [ ]:
dnabert_columns = [c for c in df_full.columns if c.startswith("emb")]
df_full = df_full.drop(columns= dnabert_columns)

In [ ]:
df_full.describe().T

## Clustering

In [ ]:
OUTDIR = Path("naiive clustering results")
OUTDIR.mkdir(parents=True, exist_ok=True)


k_values = [1, 2, 3, 4, 5]
random_state = 42
n_init = 10


df = df_full.copy()


feature_cols = ["gc_content", "length"]

# Optional ID columns (only keep the ones that actually exist)
possible_id_cols = ["phage", "Unnamed: 0", "ncbi_acc"]
id_cols_present = [c for c in possible_id_cols if c in df.columns]


feat_df = df[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)

X = feat_df.values
X_scaled = StandardScaler().fit_transform(X)

print("Feature matrix shape:", X_scaled.shape)


metrics = []
labels_by_k = {}

for k in k_values:
    km = KMeans(n_clusters=k, random_state=random_state, n_init=n_init)
    labels = km.fit_predict(X_scaled)
    labels_by_k[k] = labels

    if k == 1:
        sil = np.nan
        dbi = np.nan
    else:
        sil = silhouette_score(X_scaled, labels)
        dbi = davies_bouldin_score(X_scaled, labels)

    metrics.append({
        "k": k,
        "inertia": km.inertia_,
        "silhouette": sil,
        "davies_bouldin": dbi
    })

metrics_df = pd.DataFrame(metrics).sort_values("k").reset_index(drop=True)
print(metrics_df)


sns.set(style="whitegrid")

k_plot_values = [k for k in k_values if k >= 2]

fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)
axes = axes.ravel()

for ax, k in zip(axes, k_plot_values):
    tmp = df.copy()
    tmp["cluster"] = labels_by_k[k]

    sns.scatterplot(
        ax=ax,
        data=tmp,
        x=feature_cols[0],
        y=feature_cols[1],
        hue="cluster",
        palette="viridis",
        s=90,
        alpha=0.95,
        edgecolor=None
    )
    ax.set_title(f"KMeans on [{feature_cols[0]}, {feature_cols[1]}], k={k}")
    ax.set_xlabel(feature_cols[0])
    ax.set_ylabel(feature_cols[1])
    ax.legend(title="Cluster", loc="best")

fig.savefig(OUTDIR / "kmeans_subplots_k2_k5.png", dpi=1200, bbox_inches="tight")
plt.show()


plt.figure(figsize=(8, 5))
plt.plot(metrics_df["k"], metrics_df["inertia"], marker="o")
plt.title("Elbow Method (Inertia vs k)")
plt.xlabel("k")
plt.ylabel("Inertia")
plt.xticks(k_values)
plt.grid(True, linestyle="--", alpha=0.5)
plt.savefig(OUTDIR / "elbow_k1_to_k5.png", dpi=1200, bbox_inches="tight")
plt.show()

score_df = metrics_df[metrics_df["k"] >= 2].copy()

plt.figure(figsize=(8, 5))
plt.plot(score_df["k"], score_df["silhouette"], marker="o", label="Silhouette (higher better)")
plt.plot(score_df["k"], score_df["davies_bouldin"], marker="o", label="Davies–Bouldin (lower better)")
plt.title("Clustering Metrics vs k (k≥2)")
plt.xlabel("k")
plt.ylabel("Score")
plt.xticks(score_df["k"].tolist())
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.savefig(OUTDIR / "scores_k2_to_k5.png", dpi=1200, bbox_inches="tight")
plt.show()


# Export assignments + full df with labels

if len(id_cols_present) > 0:
    assign_df = df[[id_cols_present[0]]].copy()
    assign_df[id_cols_present[0]] = assign_df[id_cols_present[0]].astype(str)
    assign_df = assign_df.set_index(id_cols_present[0])
else:
    assign_df = pd.DataFrame(index=df.index)

for k in k_values:
    assign_df[f"k={k}"] = labels_by_k[k].astype(int)

out_xlsx = OUTDIR / "kmeans_direct_features_k1_to_k5.xlsx"
with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    assign_df.to_excel(writer, sheet_name="cluster_assignments")
    metrics_df.to_excel(writer, sheet_name="metrics", index=False)

    df_with_labels = df.copy()
    for k in k_values:
        df_with_labels[f"cluster_k{k}"] = labels_by_k[k].astype(int)

    df_with_labels.to_excel(writer, sheet_name="df_full_with_labels", index=False)

print(f"\nWrote: {out_xlsx}")